<a href="https://colab.research.google.com/github/aboettcher-sig/morocco-forests-sig/blob/main/scripts/utility/load_gee_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load GEE exports to SQLite

Builds the analysis database from the Drive CSV exports produced by
pull_gee_data (run morroco_forest_exploration_2026_08_11_1300), including
Dynamic World.

The database is built on local disk and copied to Drive at the end, because
SQLite over the Drive mount is slow for the index-heavy work here.

Stages:
1. Copy the export CSVs from Drive to local disk.
2. Per dataset: recover the identity the export flattened away. `system:index`
   is `{image_id}_{row_counter}`; `.geo` is GeoJSON `[lon, lat]`. Produce a
   real `image_id`, a parsed `date`, and `lon`/`lat` columns.
3. Concatenate each dataset's files and deduplicate once across the whole
   group. The export used overlapping year windows, so the same scene appears
   in more than one file; per-file dedup would miss those.
4. Build `{table}_unique_locs` per dataset and assign `unique_loc_id`. Exact
   float equality is valid because every timestamp of a dataset samples the
   same fixed pixel grid.
5. Build the persistent read index `(unique_loc_id, date)` on every table.
   This is the structure the time-series reads run on: it covers both "all
   rows at a location" and "that location's series in date order".
6. Verify, then copy the finished database to Drive.

The database is rebuilt from scratch on every run, so deduplication happens
once in-flight and there are no post-hoc `_unique` shadow tables.

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import os

run_name = "morroco_forest_exploration_2026_08_11_1300"

# GEE Drive table exports land in a folder named after `folder=` at Drive ROOT.
drive_export_path = f"/content/drive/MyDrive/morroco_forests/{run_name}"

# Local staging and database (built locally, copied to Drive at the end)
local_csv_path = f"/content/{run_name}"
db_path = f"/content/{run_name}.sqlite"

# Destination for the finished database on Drive
drive_db_path = f"/content/drive/MyDrive/morroco_forests/{run_name}/{run_name}.sqlite"

In [18]:
import shutil

os.makedirs(local_csv_path, exist_ok=True)

copied = 0
for item in os.listdir(drive_export_path):
    if item.endswith(".csv"):
        shutil.copy(os.path.join(drive_export_path, item), os.path.join(local_csv_path, item))
        copied += 1

print(f"Copied {copied} CSV files to {local_csv_path}")

Copied 174 CSV files to /content/morroco_forest_exploration_2026_08_11_1300


# Loader functions

In [19]:
import sqlite3
import polars as pl
import orjson
import pandas as pd


def sanitize_column_names(df):
    """SQLite-safe column names: replace ':', '.', ' ' with '_'."""
    df.columns = [col.replace(':', '_').replace('.', '_').replace(' ', '_') for col in df.columns]
    return df


def get_table_group(filename):
    """Map an export filename to its database table.

    Keys are tested in order; 'mTPI' precedes any broader terrain match so
    SRTM_mTPI_* lands in terrain_mTPI rather than a generic SRTM table.
    """
    groups = {
        "mTPI": "terrain_mTPI",
        "aspect": "terrain_aspect",
        "elevation": "terrain_elevation",
        "slope": "terrain_slope",
        "TerraClimate": "terraclimate",
        "DynamicWorld": "dynamic_world",
        "Landsat9": "landsat_9",
        "Landsat8": "landsat_8",
        "Landsat7": "landsat_7",
        "Landsat5": "landsat_5",
    }
    for key, value in groups.items():
        if key in filename:
            return value
    return None


def parse_date(image_id, table_group):
    """Parse an ISO date from image_id, per dataset naming convention.

    - landsat_*    : 'LC08_201035_20150410'                    -> third token, YYYYMMDD
    - terraclimate : '198501'                                   -> YYYYMM, first of month
    - dynamic_world: '20150627T110656_20150627T111031_T29SQV'  -> first token, YYYYMMDD prefix
    - terrain_*    : static, no date (image_id is '')
    """
    if not image_id:
        return None
    try:
        if table_group.startswith("landsat"):
            t = image_id.split("_")[2]
            return f"{t[0:4]}-{t[4:6]}-{t[6:8]}"
        if table_group == "terraclimate":
            return f"{image_id[0:4]}-{image_id[4:6]}-01"
        if table_group == "dynamic_world":
            t = image_id.split("_")[0]
            return f"{t[0:4]}-{t[4:6]}-{t[6:8]}"
    except (IndexError, ValueError):
        return None
    return None


def process_csv(file_path, table_group):
    """Read one export CSV and return a cleaned pandas DataFrame, or None.

    - image_id: `system:index` minus the trailing per-row counter that
      flatten() appends. Single-image exports (terrain) index rows as bare
      integers, so image_id is '' there.
    - date: parsed from image_id per dataset (None for terrain).
    - lon/lat: from `.geo`; GeoJSON coordinate order is [lon, lat].
    - within-file duplicates dropped (cross-file dedup happens after concat).
    """
    if os.stat(file_path).st_size == 0:
        print(f"Skipping empty file: {os.path.basename(file_path)}")
        return None

    try:
        df = pl.read_csv(file_path, infer_schema_length=1000)
    except pl.exceptions.NoDataError:
        print(f"Skipping empty file (NoDataError): {os.path.basename(file_path)}")
        return None

    if df.height == 0:
        print(f"Skipping header-only file: {os.path.basename(file_path)}")
        return None

    # image_id: strip the trailing `_{row_counter}` from system:index. Single-image
    # exports index rows as bare integers, which polars may read as Int, so cast
    # to str first; a bare counter has no image identity and yields ''.
    df = df.with_columns(pl.col("system:index").cast(pl.Utf8))
    df = df.with_columns(
        pl.col("system:index")
        .map_elements(lambda s: s.rsplit("_", 1)[0] if "_" in s else "", return_dtype=pl.Utf8)
        .alias("image_id")
    ).drop("system:index")

    # date per dataset convention
    df = df.with_columns(
        pl.col("image_id")
        .map_elements(lambda s: parse_date(s, table_group), return_dtype=pl.Utf8)
        .alias("date")
    )

    # lon/lat from .geo; GeoJSON coordinate order is [lon, lat]
    df = df.with_columns(
        pl.col(".geo").map_elements(
            lambda x: orjson.loads(x)["coordinates"] if x else [None, None],
            return_dtype=pl.List(pl.Float64)
        ).alias("coordinates")
    )
    df = df.with_columns(
        pl.col("coordinates").list.get(0).alias("lon"),
        pl.col("coordinates").list.get(1).alias("lat"),
    ).drop("coordinates", ".geo")

    df = df.unique()  # within-file duplicates
    df = df.to_pandas()
    df = sanitize_column_names(df)
    return df


def load_csv_files_to_db(folder_path, db_path):
    """Fresh-build the database, one table per dataset.

    Each dataset's files are concatenated and deduplicated once across the
    whole group (the export's overlapping year windows put the same scene in
    more than one file). Datasets that yield no rows produce no table.
    """
    if os.path.exists(db_path):
        os.remove(db_path)
        print(f"Removed existing database: {db_path}")

    grouped_files = {}
    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith(".csv"):
            group = get_table_group(filename)
            if group:
                grouped_files.setdefault(group, []).append(os.path.join(folder_path, filename))
            else:
                print(f"No table group for: {filename}")

    conn = sqlite3.connect(db_path)
    for group, files in grouped_files.items():
        frames = [process_csv(f, group) for f in files]
        frames = [f for f in frames if f is not None and not f.empty]
        if not frames:
            print(f"{group}: no data in {len(files)} file(s), skipping")
            continue
        table = pd.concat(frames, ignore_index=True)
        before = len(table)
        table = table.drop_duplicates(ignore_index=True)
        print(f"{group}: {len(files)} file(s), {before} rows -> {len(table)} after cross-file dedup")
        table.to_sql(group, conn, if_exists="replace", index=False)
    conn.commit()
    conn.close()
    print("Load complete.")

# Build the database

In [20]:
load_csv_files_to_db(local_csv_path, db_path)

Removed existing database: /content/morroco_forest_exploration_2026_08_11_1300.sqlite
Skipping empty file (NoDataError): 3DEP_morroco_forest_exploration_2026_08_11_1300_aspect.csv
terrain_aspect: 2 file(s), 303 rows -> 303 after cross-file dedup
Skipping empty file (NoDataError): 3DEP_morroco_forest_exploration_2026_08_11_1300_elevation.csv
terrain_elevation: 2 file(s), 287 rows -> 287 after cross-file dedup
Skipping empty file (NoDataError): 3DEP_morroco_forest_exploration_2026_08_11_1300_slope.csv
terrain_slope: 2 file(s), 303 rows -> 303 after cross-file dedup
dynamic_world: 1 file(s), 6769633 rows -> 6769633 after cross-file dedup
Skipping empty file (NoDataError): Landsat5_morroco_forest_exploration_2026_08_11_1300_2012_2013.csv
Skipping empty file (NoDataError): Landsat5_morroco_forest_exploration_2026_08_11_1300_2013_2014.csv
Skipping empty file (NoDataError): Landsat5_morroco_forest_exploration_2026_08_11_1300_2014_2015.csv
Skipping empty file (NoDataError): Landsat5_morroco_fo

In [21]:
# Tables and row counts
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
for table in [t[0] for t in cursor.fetchall()]:
    cursor.execute(f"SELECT COUNT(*) FROM `{table}`;")
    print(f"{table}: {cursor.fetchone()[0]} rows")
conn.close()

terrain_aspect: 303 rows
terrain_elevation: 287 rows
terrain_slope: 303 rows
dynamic_world: 6769633 rows
landsat_5: 321352 rows
landsat_7: 284929 rows
landsat_8: 708891 rows
landsat_9: 228208 rows
terrain_mTPI: 94 rows
terraclimate: 17280 rows


In [22]:
# Preview one row per data table
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
for table in [t[0] for t in cursor.fetchall()]:
    df = pd.read_sql_query(f"SELECT * FROM `{table}` LIMIT 1;", conn)
    print(f"--- {table} ---")
    print(df.iloc[0])
conn.close()

--- terrain_aspect ---
aspect       17.49807
image_id            0
date             None
lon         -5.468988
lat         35.400674
Name: 0, dtype: object
--- terrain_elevation ---
elevation    314.54437
image_id             0
date              None
lon          -5.465348
lat          35.399509
Name: 0, dtype: object
--- terrain_slope ---
slope       10.250081
image_id            0
date             None
lon         -5.471414
lat         35.399057
Name: 0, dtype: object
--- dynamic_world ---
bare                                                  0.072294
built                                                 0.064639
crops                                                 0.048676
flooded_vegetation                                    0.051296
grass                                                 0.089034
label                                                        1
shrub_and_scrub                                       0.280204
snow_and_ice                                          0.035673

# Unique locations

Within one dataset every timestamp samples the same fixed pixel grid, so
repeated visits to a location have bit-identical lon/lat and exact float
equality is a valid join. The `(lat, lon)` index on each `_unique_locs`
table is what the keying join below looks into, and it stays for
coordinate-to-location lookup at query time.

In [23]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE '%_unique_locs';")
data_tables = [t[0] for t in cursor.fetchall()]

for table in data_tables:
    cursor.execute(f"DROP TABLE IF EXISTS `{table}_unique_locs`;")
    cursor.execute(f"""
        CREATE TABLE `{table}_unique_locs` (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            lat FLOAT,
            lon FLOAT
        );
    """)
    cursor.execute(f"""
        INSERT INTO `{table}_unique_locs` (lat, lon)
        SELECT DISTINCT lat, lon FROM `{table}`
        WHERE lat IS NOT NULL AND lon IS NOT NULL;
    """)
    cursor.execute(f"CREATE INDEX IF NOT EXISTS idx_{table}_unique_locs_latlon ON `{table}_unique_locs` (lat, lon);")
    cursor.execute(f"SELECT COUNT(*) FROM `{table}_unique_locs`;")
    print(f"{table}: {cursor.fetchone()[0]} unique locations")

conn.commit()
conn.close()

terrain_aspect: 303 unique locations
terrain_elevation: 287 unique locations
terrain_slope: 303 unique locations
dynamic_world: 8734 unique locations
landsat_5: 1273 unique locations
landsat_7: 1434 unique locations
landsat_8: 1273 unique locations
landsat_9: 1273 unique locations
terrain_mTPI: 94 unique locations
terraclimate: 36 unique locations


# Assign unique_loc_id

Each data row gets the id of its location. The correlated lookup reads into
`{table}_unique_locs`, which is indexed on `(lat, lon)`.

In [24]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

for table in data_tables:
    u = f"{table}_unique_locs"
    cursor.execute(f"ALTER TABLE `{table}` ADD COLUMN unique_loc_id INTEGER;")
    cursor.execute(f"""
        UPDATE `{table}`
        SET unique_loc_id = (
            SELECT id FROM `{u}`
            WHERE `{u}`.lat = `{table}`.lat AND `{u}`.lon = `{table}`.lon
        );
    """)
    print(f"{table}: keyed by unique_loc_id")

conn.commit()
conn.close()

terrain_aspect: keyed by unique_loc_id
terrain_elevation: keyed by unique_loc_id
terrain_slope: keyed by unique_loc_id
dynamic_world: keyed by unique_loc_id
landsat_5: keyed by unique_loc_id
landsat_7: keyed by unique_loc_id
landsat_8: keyed by unique_loc_id
landsat_9: keyed by unique_loc_id
terrain_mTPI: keyed by unique_loc_id
terraclimate: keyed by unique_loc_id


# Read index for time-series retrieval

The transformer feed reads one location's series at a time:
`WHERE unique_loc_id = ? ORDER BY date`. The composite `(unique_loc_id, date)`
index serves that directly. Its leading column also covers plain
`WHERE unique_loc_id = ?` lookups (terrain tables, where date is null). This
is the persistent structure the reads run on, built last over the final state.

In [25]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

for table in data_tables:
    cursor.execute(f"CREATE INDEX IF NOT EXISTS idx_{table}_loc_date ON `{table}` (unique_loc_id, date);")
    print(f"{table}: indexed on (unique_loc_id, date)")

conn.commit()
conn.close()

terrain_aspect: indexed on (unique_loc_id, date)
terrain_elevation: indexed on (unique_loc_id, date)
terrain_slope: indexed on (unique_loc_id, date)
dynamic_world: indexed on (unique_loc_id, date)
landsat_5: indexed on (unique_loc_id, date)
landsat_7: indexed on (unique_loc_id, date)
landsat_8: indexed on (unique_loc_id, date)
landsat_9: indexed on (unique_loc_id, date)
terrain_mTPI: indexed on (unique_loc_id, date)
terraclimate: indexed on (unique_loc_id, date)


# Verification

In [26]:
# Every row in every data table should carry a unique_loc_id
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
for table in data_tables:
    cursor.execute(f"SELECT COUNT(*) FROM `{table}` WHERE unique_loc_id IS NULL;")
    nulls = cursor.fetchone()[0]
    print(f"{table}: {'OK' if nulls == 0 else f'PROBLEM: {nulls} rows unkeyed'}")
conn.close()

terrain_aspect: OK
terrain_elevation: OK
terrain_slope: OK
dynamic_world: OK
landsat_5: OK
landsat_7: OK
landsat_8: OK
landsat_9: OK
terrain_mTPI: OK
terraclimate: OK


In [27]:
# Per-location series lengths for a time-series table (should equal that
# location's number of observed dates)
conn = sqlite3.connect(db_path)
check_table = "dynamic_world"
if check_table in data_tables:
    df = pd.read_sql_query(f"""
        SELECT unique_loc_id, COUNT(*) AS n_obs, COUNT(DISTINCT date) AS n_dates
        FROM `{check_table}`
        GROUP BY unique_loc_id
        ORDER BY unique_loc_id
        LIMIT 10;
    """, conn)
    print(df)
conn.close()

   unique_loc_id  n_obs  n_dates
0              1    776      715
1              2    778      714
2              3    773      713
3              4    776      716
4              5    768      705
5              6    768      705
6              7    776      715
7              8    778      718
8              9    774      711
9             10    778      716


# Copy the database to Drive

In [28]:
os.makedirs(os.path.dirname(drive_db_path), exist_ok=True)
shutil.copy(db_path, drive_db_path)
print(f"Database copied to {drive_db_path}")

Database copied to /content/drive/MyDrive/morroco_forests/morroco_forest_exploration_2026_08_11_1300/morroco_forest_exploration_2026_08_11_1300.sqlite
